# Inferencia de una red booleana de polarización de CAF desde datos single-cell

**Autora:** Lorena Ramos Ambou · M2 CARe Bio-Santé, Université Toulouse III · NetB(IO)², CRCT

**Datos (ArrayExpress):** E-MTAB-14944 (co-cultivo PSC día 6) y E-MTAB-14945 (serie temporal de co-cultivo, días 1–6). Origen: Cumming et al. 2025, *Cancer Research*.

**Objetivo.** Obtener una red inferida desde los datos (sin fijar los atractores), los patrones de expresión y las listas de genes que caracterizan los estados finales; después compararla con la red construida desde la literatura (v4).

**Etapas.**
1. Carga de E-MTAB-14945 y actividad de TF por célula
2. Genes y TF que caracterizan cada estado
3. Orden temporal por célula (día y pseudotiempo)
4. Inferencia de la red (prior OmniPath/CollecTRI + datos)
5. Binarización y BoNesis
6. Comparación con la red de la literatura (v4)

## 0. Preparación

**Qué hace y por qué.** Instala/importa las librerías y fija la ruta de los datos en una sola variable (`DATA`), para que cualquiera que revise el repositorio solo cambie esa línea. Los datos crudos no se suben a GitHub: se descargan de ArrayExpress (E-MTAB-14944 y E-MTAB-14945).

In [ ]:
# pip install scanpy decoupler h5py pandas scipy openpyxl matplotlib   # (una vez)
from pathlib import Path
import h5py, numpy as np, pandas as pd, scipy.sparse as sp, decoupler as dc

DATA = Path(r"C:\BoolModel\DATA")     # <- única línea que hay que cambiar en otro ordenador
D15 = DATA / "E-MTAB-14945"

## 1. Carga de E-MTAB-14945 y actividad de TF por célula

**Qué hace y por qué.** Lee la matriz de conteos (formato `.h5`, grupo `unknown`), empareja cada célula con su anotación (los códigos de barras del `.h5` van en minúsculas y los de la anotación en mayúsculas, por eso se pasa todo a mayúsculas) y normaliza (CP10K + log1p). Después estima la actividad de cada factor de transcripción en **cada célula** con decoupleR (método `ulm`) y la red CollecTRI de ratón. Se trabaja por célula, no con la media del grupo, porque la media de grupo nos engañó antes (caso de SMAD3 en iCAF).

**Qué esperar.** Se imprimen las dimensiones, las columnas de la anotación y los conteos por día y por estado. Con eso se fija qué columna da el orden temporal.

In [ ]:
h5f = next(D15.rglob("*Raw_UMI_Count.h5"))
anf = next(D15.rglob("*single_cell_annotation*"))
print(h5f, "\n", anf)

with h5py.File(h5f, "r") as f:
    g = f["unknown"]
    barc  = [b.decode().upper() for b in g["barcodes"][:]]
    genes = [x.decode() for x in g["gene_names"][:]]
    M15 = sp.csc_matrix((g["data"][:], g["indices"][:], g["indptr"][:]), shape=(len(genes), len(barc)))
ann15 = pd.read_csv(anf, sep=None, engine="python")
ann15["Cell.ID"] = ann15["Cell.ID"].str.upper()
print(M15.shape, ann15.shape); print(ann15.columns.tolist()); print(ann15.head(3).to_string())

cand = [c for c in ann15.columns if any(k in c.lower() for k in ("day", "time"))]
print("columnas candidatas a día:", cand)
for c in cand + ["Annotation"]:
    if c in ann15.columns: print(ann15[c].value_counts().sort_index().to_string(), "\n")

In [ ]:
def logcpm(M):
    M = M.tocsc().astype(float); tot = np.asarray(M.sum(axis=0)).ravel(); tot[tot == 0] = 1
    N = (M @ sp.diags(1e4 / tot)).tocsc(); N.data = np.log1p(N.data); return N

pos = {b: i for i, b in enumerate(barc)}
a = ann15[ann15["Cell.ID"].isin(pos)].reset_index(drop=True)
N = logcpm(M15[:, a["Cell.ID"].map(pos).values])          # genes x células, log-CP10K
net = dc.op.collectri(organism="mouse")
keep = sorted(set(net["target"]) & set(genes)); gi = {g: i for i, g in enumerate(genes)}
df = pd.DataFrame(N[[gi[g] for g in keep], :].T.toarray().astype("float32"),
                  index=a["Cell.ID"].values, columns=keep)
tf = dc.mt.ulm(df, net)[0]                                 # actividad de TF por célula
tf.to_csv("tf_activity_14945.csv"); a.to_csv("meta_14945.csv", index=False)
print("TFs:", tf.shape)

## 2. Genes y TF que caracterizan cada estado

**Qué hace y por qué.** Para cada estado anotado compara sus células con el resto (test de Wilcoxon) y lista los genes con más expresión diferencial, y lo mismo con la actividad de TF, que es más estable que un gen suelto. Es la respuesta directa a "qué patrones y qué lista de genes caracteriza cada estado".

**Cómo leerlo.** Los marcadores de un estado frente al resto mezclan lo que lo define con lo que comparte con sus vecinos temporales; los estados finales (ifCAFs, myCAFs, iCAFs/ifCAFs) son los que más interesan, pero hay que mirar también los intermedios. Con pocas células por estado los marcadores son poco fiables, y ddSeq capta pocos genes por célula, así que los genes poco expresados (citoquinas) salen menos que los TF.

In [ ]:
# pip install scanpy   (una vez)
import scanpy as sc, anndata as ad

adata = ad.AnnData(X=N.T.tocsr(), obs=a.set_index("Cell.ID"), var=pd.DataFrame(index=genes))
adata.var_names_make_unique()
estado = "Annotation"                       # columna con los 7 estados
print(adata.obs[estado].value_counts())

sc.tl.rank_genes_groups(adata, estado, method="wilcoxon", pts=True)
res = sc.get.rank_genes_groups_df(adata, group=None)
res.to_csv("marcadores_estados_14945.csv", index=False)
for est in adata.obs[estado].unique():
    t = res[(res.group == est) & (res.pvals_adj < 0.01) & (res.logfoldchanges > 0.25)].head(15)
    print(f"\n== {est} (n={int((adata.obs[estado]==est).sum())}):", ", ".join(t.names))

atf = ad.AnnData(X=tf.loc[adata.obs_names].values.astype("float32"),
                 obs=adata.obs, var=pd.DataFrame(index=tf.columns))
sc.tl.rank_genes_groups(atf, estado, method="wilcoxon")
rt = sc.get.rank_genes_groups_df(atf, group=None); rt.to_csv("tf_estados_14945.csv", index=False)
for est in atf.obs[estado].unique():
    t = rt[(rt.group == est) & (rt.pvals_adj < 0.01)].head(10)
    print(f"{est:14s} TF:", ", ".join(t.names))

## 3–6. Pendientes

3. Orden temporal por célula (día y pseudotiempo)
4. Inferencia de la red (prior OmniPath/CollecTRI + datos)
5. Binarización y BoNesis
6. Comparación con la red de la literatura (v4)